# Working With Files

Data is spread all over the internet, your own system and everywhere else. To work with it, you need to read it into your program. This section will cover how to read data from files, and how to write data back to files.

## Setup

Before you can start you maybe need to install some packages if not already done.

In [ ]:
%pip install -r ../requirements.txt

import csv # Working with CSV files
import psutil # Getting system stats like CPU, Memory, Disk and Battery usage
import time # Working with time, e.g. for sleeping between collecting system stats
import json # Working with JSON files
import yaml # Working with YAML files
import xml.etree.ElementTree as et # Working with XML files

from xml.dom import minidom # Working with XML files, e.g. for pretty printing XML data

# Working with text files

Some files have a specific format. Others are just plain text files. To read and write to these files, you can use the built-in `open` function. This function takes the file path and the mode (read, write, append) as arguments. `open` has some parameters you need to know:

- `file`: The path to the file you want to open. The path can be absolute (e.g., 'C:/data/file.txt') or relative (e.g., '../data/file.txt').
- `mode`: The mode in which you want to open the file. Common modes include:
  - `'r'`: Read mode (default). Opens the file for reading.
  - `'w'`: Write mode. Opens the file for writing (creates a new file or overwrites an existing file).
  - `'a'`: Append mode. Opens the file for writing (creates a new file or appends to an existing file).
- `encoding`: The encoding of the file (e.g., 'utf-8'). If the file you dealing with is not some other encoding, you should mostly use 'utf-8' for text files to ensure compatibility with different characters.

> [!IMPORTANT]
> Always remember to close the file after you're done to free up system resources. You can do this manually with `file.close()` or use a context manager (`with` statement) which automatically handles closing the file for you.

In [ ]:
with open('../assets/texts/strawberry_ice_cream.md','r', encoding='utf-8') as file:
    print(file.read())

If the file is not too big, you can read the whole content at once using `file.read()`. For larger files, you can read it line by line using a loop or `file.readline()`. There is also `file.readlines()` which reads all lines into a list.

In [ ]:
with open('../assets/texts/strawberry_ice_cream.md','r', encoding='utf-8') as file:
    for line in file:
        print(line.strip())

with open('../assets/texts/strawberry_ice_cream.md','r', encoding='utf-8') as file:
    print(file.readlines())

With `open`, you can also write to files. You can use `file.write()` to write a string to the file. If you want to write multiple lines, you can use `file.writelines()` which takes a list of strings and writes them to the file.

In [ ]:
with open('../temp/test.txt','w', encoding='utf-8') as file:
    file.write('This is a test.\n')

## Working With Specific File Formats

For files with other formats you should use specific libraries to read and write to them. Here are some useful libraries to handle them:

- for CSV files you can use the `csv` module
- for JSON files you can use the `json` module
- for XML files you can use the `xml.etree.ElementTree` module
- for YAML files you can use the `PyYAML` library

These are all very common plain text formats, so I want to show you some little examples for working with them.

All those formats offer ability to structure the data you working with. All have there strength and weaknesses for different use cases.

## CSV

CSV is a simple format for tabular data. It is easy to read and write, but it does not support complex data structures or nested data. For working with the data stored in CSV files, there might be better options like `pandas` which can handle CSV files and provide powerful data manipulation capabilities. But if you are collecting data and want to store it the lib `csv` is a good option.

You can read more about the format at [RFC 4180: Common Format and MIME Type for Comma-Separated Values (CSV) Files](https://www.rfc-editor.org/rfc/rfc4180).
The documentation for the `csv` module can be found at [csv — CSV File Reading and Writing](https://docs.python.org/3/library/csv.html).

In [ ]:
data = [['CPU', 'Memory', 'Disk', 'Battery']]

# Collects system stats every second for 10 seconds.
for _ in range(10):
    data.append([psutil.cpu_percent(), psutil.virtual_memory().percent, psutil.disk_usage('/').percent, psutil.sensors_battery().percent if psutil.sensors_battery() else None])
    time.sleep(1)

# Writes the data as CSV to a file.
# Look into the temp folder to see the file. You can open it with Excel or any text editor.
with open('../temp/system_stats.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quotechar="\"", delimiter=',', quoting=csv.QUOTE_MINIMAL)
    writer.writerows(data)

# Reads the CSV file and prints each row.
with open('../temp/system_stats.csv', 'r', encoding='utf-8') as file:
    reader = csv.reader(file, delimiter=',', quotechar="\"")
    for row in reader:
        print(row) 

## JSON

JSON is a popular format for data exchange. It is easy to read and write, and it supports complex data structures and nested data. It is widely used in web applications and APIs. JSON stands for Javascript Object Notation, but it is language independent and can be used in many programming languages. But it's syntax is based on JavaScript, so working with it in JavaScript based front-ends is very easy. JSON is a good choice if you are dealing with API data, data loaded from some kind of NoSQL database or if you want to store data in a structured way in files on disk. It can be minified to save space.

If you have a Python dictionary it will perfectly map to a JSON object. If you have a Python list it will perfectly map to a JSON array. So you can easily convert between Python data structures and JSON.

You can read more about the format at [RFC 8259 - The JavaScript Object Notation (JSON) Data Interchange Format](https://datatracker.ietf.org/doc/html/rfc8259).
The documentation for the `json` module can be found at [json — JSON Encoder and Decoder](https://docs.python.org/3/library/json.html).

In [ ]:
data = []

for _ in range(10):
    data.append({
        'CPU': psutil.cpu_percent(),
        'Memory': psutil.virtual_memory().percent,
        'Disk': psutil.disk_usage('/').percent,
        'Battery': psutil.sensors_battery().percent if psutil.sensors_battery() else None,
        'Net': {
            'Bytes Sent': psutil.net_io_counters().bytes_sent,
            'Bytes Received': psutil.net_io_counters().bytes_recv
        }
    })

    time.sleep(1)

# Writes the data as JSON to a file. Look into the temp folder to see the file. You can open it with any text editor.
with open('../temp/system_stats.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4)

# Reads the JSON file and prints each entry.
with open('../temp/system_stats.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
    
    for entry in data:
        print(entry)

## XML

XML is very old and was once the most popular format for data exchange. It is still used in some legacy systems and in some specific use cases, but it has largely been replaced by JSON. A famous variance of XML is HTML which is used for structuring web pages. XAML is a desktop UI markup language based on XML. XML has opening and closing tags, which increase the size of the data compared to JSON, but it also has an advantage. XML tags can have attributes, which can be used to store additional information about the data.

If you don't have a good reason to use XML, you should probably avoid it. Parsing HTML or working with legacy data might be the only reason to use it.

You can read more about the format at [Extensible Markup Language (XML) 1.0 (Fifth Edition)](https://www.w3.org/TR/REC-xml/).
The documentation for the `xml.etree.ElementTree` module can be found at [xml.etree.ElementTree — The ElementTree XML API — Python 3.14.5rc1 documentation](https://docs.python.org/3/library/xml.etree.elementtree.html#module-xml.etree.ElementTree).

In [ ]:
# Creates an XML structure and writes it to a file. Look into the temp folder to see the file. You can open it with any text editor.
root = et.Element('SystemStats')

for entry in data:
    stat = et.SubElement(root, 'Stat')
    stat.set('timestamp', str(time.time()))
    
    cpu = et.SubElement(stat, 'CPU')
    cpu.text = str(entry['CPU'])
    
    memory = et.SubElement(stat, 'Memory')
    memory.text = str(entry['Memory'])
    
    disk = et.SubElement(stat, 'Disk')
    disk.text = str(entry['Disk'])
    
    battery = et.SubElement(stat, 'Battery')
    battery.text = str(entry['Battery'])
    
    net = et.SubElement(stat, 'Net')
    
    bytes_sent = et.SubElement(net, 'BytesSent')
    bytes_sent.text = str(entry['Net']['Bytes Sent'])
    
    bytes_recv = et.SubElement(net, 'BytesReceived')
    bytes_recv.text = str(entry['Net']['Bytes Received'])

# Writes the XML structure to a file. Look into the temp folder to see the file. You can open it with any text editor.
tree = et.ElementTree(root)
tree.write('../temp/system_stats.xml', encoding='utf-8', xml_declaration=True)

# Reads the XML file and prints it pretty printed.
with open('../temp/system_stats.xml', 'r', encoding='utf-8') as file:
    xml_content = file.read()

pretty_xml = minidom.parseString(xml_content).toprettyxml(indent='  ')
print(pretty_xml)

## YAML

YAML is a human-readable data serialization format. It is often used for configuration files and in applications where data is being stored or transmitted. YAML stands for "YAML Ain't Markup Language". It is designed to be easy to read and write, and it supports complex data structures and nested data. YAML uses indentation to indicate nesting, which makes it visually appealing and easy to understand. It also supports comments, which can be helpful for documenting the data.

You can use YAML as a more human-friendly alternative to JSON. Spaces are part of the syntax, so you can not minify it like JSON. If the size matters use JSON. If you want to read the data use YAML. It is often used for configuration files.

Let's use the data from the JSON example and write it to a YAML file.

You can read more about the format at [YAML Ain't Markup Language (YAML™) Version 1.2](https://yaml.org/spec/1.2/spec.html).
The documentation for the `PyYAML` library can be found at [PyYAML Documentation](https://pyyaml.org/wiki/PyYAMLDocumentation).

In [ ]:
# Writes the data as YAML to a file. Look into the temp folder to see the file. You can open it with any text editor.
with open('../temp/system_stats.yaml', 'w', encoding='utf-8') as file:
    yaml.dump(data, file, indent=4)

# Reads the YAML file and prints each entry.
with open('../temp/system_stats.yaml', 'r', encoding='utf-8') as file:
    data = yaml.safe_load(file)
    
    for entry in data:
        print(entry)